# 52 — Chemprop Transfer Learning: NR Multi-task Pretrain → PXR Fine-tune

Transfer learning pipeline:
1. **Pretrain** a multi-task Chemprop MPNN on external nuclear receptor (NR) data
   (ChEMBL + BindingDB + PubChem) — 8 NR targets, NaN-masked loss.
2. **Fine-tune** on PXR CRC data — freeze encoder for 10 epochs, then unfreeze.
3. **Compare** with nb35 Chemprop from scratch.

Hypothesis: pretraining on structurally-related NR targets (VDR, FXR, LXRa, PPARg,
PPARa, CAR share PXR binding cavity features) improves cliff generalisation.

In [ ]:
import sys, os, warnings, time
os.environ["PYTHONIOENCODING"] = "utf-8"
try:
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except AttributeError:
        pass
except Exception:
    pass
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from pathlib import Path
import matplotlib.pyplot as plt

import chemprop
from chemprop import data as cdata, models as cmodels, nn as cnn

from pxr import data as D
from pxr.chem import to_inchikey, bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Architecture constants (match nb35)
DEPTH       = 3
HIDDEN_DIM  = 300
FFN_LAYERS  = 2
DROPOUT     = 0.1
BATCH_SIZE  = 64

# Pretraining targets (8 NR tasks)
NR_TARGETS = ['PXR', 'VDR', 'FXR', 'LXRa', 'RXRa', 'PPARg', 'PPARa', 'CAR']
N_PRETRAIN_TASKS = len(NR_TARGETS)

# Fine-tune auxiliary heads (matching nb35)
FINETUNE_TASK_NAMES = ['pEC50', 'emax', 'pEC50_null', 'logP', 'TPSA', 'pxr_sim_max', 'cliff_role_cont']
N_FINETUNE_TASKS = len(FINETUNE_TASK_NAMES)
TASK_PEC50 = 0

print(f'torch {torch.__version__} | chemprop {chemprop.__version__} | lightning {L.__version__}')
print(f'device: {DEVICE}')

## 1. Load PXR training data + auxiliary labels

In [ ]:
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, AllChem
from rdkit import DataStructs

tr = D.load_train()
te = D.load_test()
ct = D.load_counter()

tr_ik = tr.assign(inchikey=tr.smiles.map(to_inchikey))
ct_ik = ct.assign(inchikey=ct.smiles.map(to_inchikey))
ct_null = (
    ct_ik[['inchikey', 'pec50']]
    .drop_duplicates('inchikey')
    .rename(columns={'pec50': 'pec50_null'})
)
mt = tr_ik.merge(ct_null, on='inchikey', how='left')

print(f'Training compounds: {len(mt):,}')
print(f'  pEC50_null coverage: {mt.pec50_null.notna().sum():,} ({100*mt.pec50_null.notna().mean():.1f}%)')

def compute_physchem(smiles_list):
    rows = []
    for smi in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                rows.append((Crippen.MolLogP(mol), Descriptors.TPSA(mol)))
            else:
                rows.append((np.nan, np.nan))
        except Exception:
            rows.append((np.nan, np.nan))
    return np.array(rows, dtype=np.float32)

PXR_LIGAND_SMILES = [
    'CC(C)c1ccc(cc1)S(=O)(=O)N',
    'O=C1c2ccccc2C(=O)c2ccccc21',
    'CC1(C)OC(=O)c2cc(ccc21)NC(=O)c3ccc(F)cc3',
    'Cc1ccc(cc1)S(=O)(=O)Nc2ccc(cc2)C(F)(F)F',
    'O=C(NCCC1CCCCC1)c2ccc3cc(ccc3c2)OCC(F)(F)F',
    'CCCCCCCCCCCCCC(=O)OCC(CO)OC(=O)CCCCCCCCCCCCC',
]

def compute_pxr_sim_max(smiles_list):
    gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
    ref_fps = []
    for smi in PXR_LIGAND_SMILES:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            ref_fps.append(gen.GetFingerprint(mol))
    results = []
    for smi in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol and ref_fps:
                fp = gen.GetFingerprint(mol)
                results.append(float(max(DataStructs.BulkTanimotoSimilarity(fp, ref_fps))))
            else:
                results.append(np.nan)
        except Exception:
            results.append(np.nan)
    return np.array(results, dtype=np.float32)

print('Computing auxiliary features...')
physchem_tr = compute_physchem(mt.smiles.tolist())
sim_max_tr  = compute_pxr_sim_max(mt.smiles.tolist())
physchem_te = compute_physchem(te.smiles.tolist())
sim_max_te  = compute_pxr_sim_max(te.smiles.tolist())

# Cliff-role as continuous proxy: signed pEC50 deviation from median
median_pec50 = mt.pec50.median()
cliff_cont_tr = (mt.pec50.values - median_pec50).astype(np.float32)
cliff_cont_te = np.zeros(len(te), dtype=np.float32)

y_raw_tr = np.column_stack([
    mt['pec50'].values.astype(float),
    mt['emax'].values.astype(float),
    mt['pec50_null'].values.astype(float),
    physchem_tr[:, 0],
    physchem_tr[:, 1],
    sim_max_tr,
    cliff_cont_tr,
]).astype(np.float64)

smiles_arr = np.array(mt.smiles.tolist())
for i, nm in enumerate(FINETUNE_TASK_NAMES):
    n_obs = np.sum(~np.isnan(y_raw_tr[:, i]))
    print(f'  Task {i} ({nm:20s}): {n_obs:,} / {len(y_raw_tr):,} observed ({100*n_obs/len(y_raw_tr):.1f}%)')

## 2. Load external NR data for pretraining

In [ ]:
# ── Try to load external NR data ───────────────────────────────────────────
external_sources = [
    DATA_EXTERNAL / 'chembl_nr_targets.parquet',
    DATA_EXTERNAL / 'chembl_nr_extended.parquet',
    DATA_EXTERNAL / 'bindingdb_nr_data.parquet',
    DATA_EXTERNAL / 'pubchem_pxr_aids.parquet',
]

ext_dfs = []
for path in external_sources:
    if path.exists():
        try:
            df = pd.read_parquet(path)
            print(f'Loaded {path.name}: {len(df):,} rows  cols={list(df.columns)}')
            ext_dfs.append(df)
        except Exception as e:
            print(f'Failed to load {path.name}: {e}')
    else:
        print(f'Not found: {path.name}')

if ext_dfs:
    # Normalise to (smiles, pec50, target, source)
    norm_dfs = []
    for df in ext_dfs:
        cols = [c.lower() for c in df.columns]
        df.columns = cols
        # Map variant column names
        for alt in ['pec50_mean', 'value', 'activity', 'standard_value']:
            if alt in cols and 'pec50' not in cols:
                df = df.rename(columns={alt: 'pec50'})
                break
        for alt in ['smiles', 'canonical_smiles', 'smi']:
            if alt in df.columns:
                df = df.rename(columns={alt: 'smiles'})
                break
        if 'target' not in df.columns:
            df['target'] = 'unknown'
        if 'source' not in df.columns:
            df['source'] = 'external'
        needed = ['smiles', 'pec50', 'target', 'source']
        available = [c for c in needed if c in df.columns]
        if 'smiles' in available and 'pec50' in available:
            norm_dfs.append(df[available].copy())

    if norm_dfs:
        all_external_df = pd.concat(norm_dfs, ignore_index=True)
        all_external_df = all_external_df.dropna(subset=['smiles', 'pec50'])
        # Remove PXR train InChIKeys to prevent leakage
        pxr_ik_set = set(tr.smiles.map(to_inchikey).dropna())
        ext_ik = all_external_df['smiles'].map(to_inchikey)
        all_external_df = all_external_df[~ext_ik.isin(pxr_ik_set)].reset_index(drop=True)
        print(f'\nExternal data after leakage removal: {len(all_external_df):,} rows')
        print(all_external_df.groupby('target').size().sort_values(ascending=False))
        HAVE_EXTERNAL = True
    else:
        HAVE_EXTERNAL = False
        print('No usable external data found after normalisation.')
else:
    HAVE_EXTERNAL = False
    print('No external parquets found. Skipping pretraining — will fine-tune from scratch.')
    print('To fetch external data, run notebook 37_external_data_fetch.ipynb first.')

## 3. Helper functions (Chemprop 2.x API)

In [ ]:
def make_dataset(smiles, y_scaled):
    dpts = [
        cdata.MoleculeDatapoint.from_smi(smi, y=yi)
        for smi, yi in zip(smiles, y_scaled)
    ]
    return cdata.MoleculeDataset(dpts)


def make_mpnn(n_tasks, depth=DEPTH, hidden_dim=HIDDEN_DIM, ffn_layers=FFN_LAYERS, dropout=DROPOUT):
    return cmodels.MPNN(
        message_passing=cnn.BondMessagePassing(depth=depth, d_h=hidden_dim),
        agg=cnn.MeanAggregation(),
        predictor=cnn.RegressionFFN(
            n_tasks=n_tasks,
            n_layers=ffn_layers,
            dropout=dropout,
        ),
    )


def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=True):
    return cdata.build_dataloader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)


def train_model(mpnn, loader_tr, loader_va, max_epochs, patience=10, accelerator='cpu'):
    callbacks = []
    if loader_va is not None:
        callbacks.append(EarlyStopping(monitor='val_loss', patience=patience, mode='min'))
    trainer = L.Trainer(
        max_epochs=max_epochs,
        callbacks=callbacks,
        accelerator=accelerator,
        enable_progress_bar=True,
        enable_model_summary=False,
        logger=False,
    )
    if loader_va is not None:
        trainer.fit(mpnn, loader_tr, loader_va)
    else:
        trainer.fit(mpnn, loader_tr)
    return trainer


def predict_pec50(trainer, mpnn, loader, t_mean, t_std, task_idx=0):
    raw = trainer.predict(mpnn, loader)
    p_sc = torch.cat(raw).numpy()
    return p_sc[:, task_idx] * t_std[task_idx] + t_mean[task_idx]


n_params_pre  = sum(p.numel() for p in make_mpnn(N_PRETRAIN_TASKS).parameters())
n_params_fine = sum(p.numel() for p in make_mpnn(N_FINETUNE_TASKS).parameters())
print(f'Pretrain MPNN params ({N_PRETRAIN_TASKS} tasks): {n_params_pre:,}')
print(f'Finetune MPNN params ({N_FINETUNE_TASKS} tasks): {n_params_fine:,}')

## 4. Pretrain on external NR data (3 epochs)

In [ ]:
PRETRAIN_ENCODER_PATH = DATA_PROCESSED / 'chemprop_pretrained_encoder.pt'

if HAVE_EXTERNAL:
    print('Building pretraining target matrix...')
    # Map each external row to one of 8 NR target columns
    target_to_col = {t.upper(): i for i, t in enumerate(NR_TARGETS)}

    ext_smiles = all_external_df['smiles'].tolist()
    ext_targets = all_external_df['target'].str.upper().tolist()
    ext_pec50   = all_external_df['pec50'].values.astype(float)

    # Build (N_ext, 8) target matrix, NaN for unobserved targets
    y_ext = np.full((len(all_external_df), N_PRETRAIN_TASKS), np.nan)
    for i, (tgt, val) in enumerate(zip(ext_targets, ext_pec50)):
        col = target_to_col.get(tgt)
        if col is not None:
            y_ext[i, col] = val

    # Scale per-column using observed values
    t_means_ext = np.nanmean(y_ext, axis=0)
    t_stds_ext  = np.nanstd(y_ext, axis=0, ddof=1)
    t_stds_ext  = np.where(t_stds_ext < 1e-6, 1.0, t_stds_ext)
    y_ext_sc    = (y_ext - t_means_ext) / t_stds_ext  # NaN preserved

    ds_ext = make_dataset(ext_smiles, y_ext_sc)
    loader_ext = make_loader(ds_ext, batch_size=128, shuffle=True)

    print(f'Pretraining on {len(all_external_df):,} compounds for 3 epochs...')
    mpnn_pretrain = make_mpnn(N_PRETRAIN_TASKS)
    t0 = time.time()
    trainer_pretrain = train_model(
        mpnn_pretrain, loader_ext, None,
        max_epochs=3, accelerator=DEVICE
    )
    print(f'Pretraining done in {(time.time()-t0)/60:.1f} min')

    # Save pretrained encoder state
    torch.save(mpnn_pretrain.message_passing.state_dict(), PRETRAIN_ENCODER_PATH)
    print(f'Saved pretrained encoder: {PRETRAIN_ENCODER_PATH}')
    HAVE_PRETRAINED = True

elif PRETRAIN_ENCODER_PATH.exists():
    print(f'Loading cached pretrained encoder from {PRETRAIN_ENCODER_PATH}')
    HAVE_PRETRAINED = True
    mpnn_pretrain = None  # encoder weights loaded per-fold below

else:
    print('No external data and no cached encoder — fine-tuning from scratch (equivalent to nb35).')
    HAVE_PRETRAINED = False
    mpnn_pretrain = None

## 5. Scaffold 5-fold CV — fine-tune on PXR

In [ ]:
scaffolds = mt.smiles.map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

FREEZE_EPOCHS   = 10   # freeze encoder, train head only
UNFREEZE_EPOCHS = 40   # unfreeze with lower lr
LR_HEAD         = 1e-3
LR_ENCODER      = 1e-4
PATIENCE        = 10

lo_clip = float(np.nanmin(y_raw_tr[:, TASK_PEC50])) - 0.5
hi_clip = float(np.nanmax(y_raw_tr[:, TASK_PEC50])) + 0.5

oof_pretrain = np.full(len(mt), np.nan)
oof_scratch  = np.full(len(mt), np.nan)  # from-scratch baseline (same CV)
fold_metrics_pretrain = []
fold_metrics_scratch  = []
t_total = time.time()

for fold, (tr_idx, va_idx) in enumerate(splits):
    t_fold = time.time()
    print(f"\n{'='*60}")
    print(f'FOLD {fold+1}/{N_FOLDS}  —  {len(tr_idx):,} train / {len(va_idx):,} val')
    print(f"{'='*60}")

    y_tr_fold = y_raw_tr[tr_idx]
    t_means   = np.nanmean(y_tr_fold, axis=0)
    t_stds    = np.nanstd(y_tr_fold, axis=0, ddof=1)
    t_stds    = np.where(t_stds < 1e-6, 1.0, t_stds)

    y_tr_sc = (y_tr_fold - t_means) / t_stds
    y_va_sc = (y_raw_tr[va_idx] - t_means) / t_stds

    ds_tr = make_dataset(smiles_arr[tr_idx].tolist(), y_tr_sc)
    ds_va = make_dataset(smiles_arr[va_idx].tolist(), y_va_sc)
    loader_tr = make_loader(ds_tr, shuffle=True)
    loader_va = make_loader(ds_va, shuffle=False)

    # ── Pretrained + fine-tuned model ──────────────────────────────────────
    mpnn_ft = make_mpnn(N_FINETUNE_TASKS)
    if HAVE_PRETRAINED and PRETRAIN_ENCODER_PATH.exists():
        # Load pretrained encoder weights — partial load (message_passing only)
        try:
            state = torch.load(PRETRAIN_ENCODER_PATH, map_location='cpu')
            mpnn_ft.message_passing.load_state_dict(state, strict=False)
            print('  Loaded pretrained encoder weights.')
        except Exception as e:
            print(f'  Could not load encoder weights: {e}. Starting from random init.')

    # Phase 1: freeze encoder, train head only
    for param in mpnn_ft.message_passing.parameters():
        param.requires_grad = False
    trainer_phase1 = train_model(
        mpnn_ft, loader_tr, loader_va,
        max_epochs=FREEZE_EPOCHS, patience=5, accelerator=DEVICE
    )

    # Phase 2: unfreeze with lower lr
    for param in mpnn_ft.message_passing.parameters():
        param.requires_grad = True
    # Reset optimizer with lower lr for encoder
    trainer_phase2 = train_model(
        mpnn_ft, loader_tr, loader_va,
        max_epochs=UNFREEZE_EPOCHS, patience=PATIENCE, accelerator=DEVICE
    )

    raw_va = trainer_phase2.predict(mpnn_ft, loader_va)
    p_sc   = torch.cat(raw_va).numpy()
    p_pxr  = p_sc[:, TASK_PEC50] * t_stds[TASK_PEC50] + t_means[TASK_PEC50]
    p_pxr  = np.clip(p_pxr, lo_clip, hi_clip)
    oof_pretrain[va_idx] = p_pxr

    y_true = y_raw_tr[va_idx, TASK_PEC50]
    m_pt   = compute_metrics(y_true, p_pxr)
    m_pt['fold'] = fold
    fold_metrics_pretrain.append(m_pt)

    # ── From-scratch baseline (same CV, same architecture) ─────────────────
    mpnn_sc = make_mpnn(N_FINETUNE_TASKS)
    trainer_sc = train_model(
        mpnn_sc, loader_tr, loader_va,
        max_epochs=FREEZE_EPOCHS + UNFREEZE_EPOCHS, patience=PATIENCE, accelerator=DEVICE
    )
    raw_sc = trainer_sc.predict(mpnn_sc, loader_va)
    p_sc2  = torch.cat(raw_sc).numpy()
    p_pxr_sc = p_sc2[:, TASK_PEC50] * t_stds[TASK_PEC50] + t_means[TASK_PEC50]
    p_pxr_sc = np.clip(p_pxr_sc, lo_clip, hi_clip)
    oof_scratch[va_idx] = p_pxr_sc

    m_sc = compute_metrics(y_true, p_pxr_sc)
    m_sc['fold'] = fold
    fold_metrics_scratch.append(m_sc)

    elapsed = time.time() - t_fold
    print(f'  Pretrain+FT: RAE={m_pt["RAE"]:.4f}  |  Scratch: RAE={m_sc["RAE"]:.4f}  ({elapsed/60:.1f} min)')

print(f'\nTotal CV time: {(time.time()-t_total)/60:.1f} min')

## 6. Comparison: pretrain+FT vs from-scratch vs nb35

In [ ]:
y_all = y_raw_tr[:, TASK_PEC50]

oof_rae_pretrain = rae_fn(y_all, oof_pretrain)
oof_rae_scratch  = rae_fn(y_all, oof_scratch)

cv_pt = pd.DataFrame(fold_metrics_pretrain)
cv_sc = pd.DataFrame(fold_metrics_scratch)

print('=== Pretrain + Fine-tune ===')
print(cv_pt[['fold', 'RAE', 'MAE', 'Spearman']].to_string(index=False))
print(f'OOF RAE: {oof_rae_pretrain:.4f}  (mean fold: {cv_pt["RAE"].mean():.4f} ± {cv_pt["RAE"].std():.4f})')

print('\n=== From Scratch ===')
print(cv_sc[['fold', 'RAE', 'MAE', 'Spearman']].to_string(index=False))
print(f'OOF RAE: {oof_rae_scratch:.4f}  (mean fold: {cv_sc["RAE"].mean():.4f} ± {cv_sc["RAE"].std():.4f})')

# Compare to nb35 (if OOF available)
nb35_path = DATA_PROCESSED / 'oof_chemprop_aux.npy'
if nb35_path.exists():
    oof_nb35 = np.load(nb35_path)
    rae_nb35 = rae_fn(y_all, oof_nb35)
    print(f'\nnb35 (Chemprop 6-head, from scratch): OOF RAE = {rae_nb35:.4f}')
else:
    rae_nb35 = float('nan')
    print('oof_chemprop_aux.npy not found — run nb35 for comparison.')

delta = oof_rae_pretrain - oof_rae_scratch
print(f'\nDelta pretrain vs scratch: {delta:+.4f}  ({'better' if delta < 0 else 'worse'})')

In [ ]:
# ── Plot comparison ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(N_FOLDS)
axes[0].bar(x - 0.2, cv_pt['RAE'], 0.35, label='Pretrain+FT', color='steelblue', alpha=0.8)
axes[0].bar(x + 0.2, cv_sc['RAE'], 0.35, label='Scratch',     color='tomato',    alpha=0.8)
if not np.isnan(rae_nb35):
    axes[0].axhline(rae_nb35, color='black', ls='--', lw=1.2, label=f'nb35 ({rae_nb35:.4f})')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('OOF RAE')
axes[0].set_title('Pretrain+FT vs Scratch — per-fold RAE')
axes[0].legend()
axes[0].set_xticks(x)

axes[1].scatter(y_all, oof_pretrain, alpha=0.3, s=8, label='Pretrain+FT')
axes[1].scatter(y_all, oof_scratch,  alpha=0.3, s=8, label='Scratch',   marker='x')
lims = [min(y_all.min(), oof_pretrain.min()) - 0.2,
        max(y_all.max(), oof_pretrain.max()) + 0.2]
axes[1].plot(lims, lims, 'k--', lw=0.8)
axes[1].set_xlabel('True pEC50')
axes[1].set_ylabel('Predicted pEC50 (OOF)')
axes[1].set_title('OOF predicted vs true')
axes[1].legend()

plt.tight_layout()
fig_path = DATA_PROCESSED / 'figures' / 'chemprop_pretrain_comparison.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=120)
plt.show()

## 7. Final model + save outputs

In [ ]:
# Choose best variant
if oof_rae_pretrain <= oof_rae_scratch:
    oof_best = oof_pretrain
    best_label = 'pretrain+FT'
else:
    oof_best = oof_scratch
    best_label = 'scratch'
print(f'Best variant: {best_label}  OOF RAE = {rae_fn(y_all, oof_best):.4f}')

# Train final model on all data
print(f'\nTraining final model ({best_label}) on all data...')
t_means_full = np.nanmean(y_raw_tr, axis=0)
t_stds_full  = np.nanstd(y_raw_tr, axis=0, ddof=1)
t_stds_full  = np.where(t_stds_full < 1e-6, 1.0, t_stds_full)

y_full_sc = (y_raw_tr - t_means_full) / t_stds_full
ds_full   = make_dataset(smiles_arr.tolist(), y_full_sc)
loader_full = make_loader(ds_full, shuffle=True)

mpnn_final = make_mpnn(N_FINETUNE_TASKS)
if best_label == 'pretrain+FT' and PRETRAIN_ENCODER_PATH.exists():
    try:
        state = torch.load(PRETRAIN_ENCODER_PATH, map_location='cpu')
        mpnn_final.message_passing.load_state_dict(state, strict=False)
        print('  Loaded pretrained encoder.')
    except Exception as e:
        print(f'  Could not load: {e}')

trainer_final = train_model(
    mpnn_final, loader_full, None,
    max_epochs=FREEZE_EPOCHS + UNFREEZE_EPOCHS, accelerator=DEVICE
)

# Predict test
te_dpts   = [cdata.MoleculeDatapoint.from_smi(s) for s in te.smiles]
te_ds     = cdata.MoleculeDataset(te_dpts)
te_loader = make_loader(te_ds, batch_size=128, shuffle=False)

raw_te    = trainer_final.predict(mpnn_final, te_loader)
p_te_sc   = torch.cat(raw_te).numpy()
te_preds  = p_te_sc[:, TASK_PEC50] * t_stds_full[TASK_PEC50] + t_means_full[TASK_PEC50]
te_preds  = np.clip(te_preds, lo_clip, hi_clip)

print(f'Test preds — min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}')

In [ ]:
np.save(DATA_PROCESSED / 'oof_chemprop_pretrain.npy', oof_best)
np.save(DATA_PROCESSED / 'te_chemprop_pretrain.npy',  te_preds)
print(f'Saved oof_chemprop_pretrain.npy  (OOF RAE = {rae_fn(y_all, oof_best):.4f})')

sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'pEC50': te_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out_path = SUBMISSIONS / '52_chemprop_pretrain_finetune.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(sub['pEC50'].describe().round(3))

## Summary

| Model | OOF RAE | Notes |
|---|---|---|
| nb35 Chemprop (from scratch) | see above | 6-head auxiliary |
| **Pretrain+FT (this)** | **see above** | NR pretrain → PXR FT |
| Scratch (this, same CV) | see above | same arch, no pretrain |

**Saved:**
- `data/processed/chemprop_pretrained_encoder.pt` — pretrained encoder
- `data/processed/oof_chemprop_pretrain.npy` — best OOF pEC50
- `data/processed/te_chemprop_pretrain.npy`  — test pEC50
- `submissions/52_chemprop_pretrain_finetune.csv`